# NB_Silver_Order_Items — Bronze to Silver Transformation (Collaborative Review & Upgrade)

## Section 1 — Purpose, Business Context & Review Legend

- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.order_items` → ADLS Gen2 / OneLake Shortcut (`Files/bronze/sales/order_items/`).
- **Silver Target:** `silver.order_items` (Managed Delta Table) + `silver._quarantine_order_items` (Conditional Quarantine).
- **Grain:** One row per order line item — composite primary key `(order_id, order_item_id)`.
- **Verified Bronze Profile (`112,650` rows × `7` columns):**
  - `0` NULLs across all 7 columns; `0` exact duplicates; `0` duplicate `(order_id, order_item_id)` composite keys.
  - `98,666` distinct `order_id` (`775` orders in `orders` have no items, mostly `unavailable`/`canceled`); `32,951` distinct `product_id`; `3,095` distinct `seller_id`; **`0` orphan keys** against `orders`, `products`, or `sellers`.
  - `order_item_id` ranges `1..21` (`0` sequence gaps). Note: `6,968` orders repeat the same `product_id` across multiple `order_item_id` lines (Olist represents quantity via separate line numbers — never deduplicate by `(order_id, product_id)`).
  - Financials: `SUM(price) = 13,591,643.70` (`0.85..6,735.00`), `SUM(freight_value) = 2,251,909.54` (`0.00..409.68`), Combined Gross Merchandise + Freight = **`15,843,553.24`**. `383` rows have `freight_value = 0.00` (valid free shipping).
  - Temporal anomaly: **`4` rows** (`3` orders from March–May 2017, including a `delivered` order) carry corrupted `shipping_limit_date` timestamps in **`2020`** (`+1,051` to `+1,056` days). Preserved in Silver with `shipping_limit_outlier_flag = True` rather than dropped so financial totals remain exact.

### Collaborative Code-Review Legend
This notebook preserves the original teammate draft (`order_items.ipynb`) alongside production upgrades so every design decision is transparent:
- **`[ORIGINAL — COMMENTED FOR REVIEW]`**: Preserved verbatim from the teammate's notebook inside comments (inactive so `Run All` never fails).
- **`[IMPROVED REPLACEMENT]`**: Production Spark replacement directly beneath the commented original cell, with rationale.
- **`[KEPT / BUILT UPON]`**: Sound original logic (e.g., `DecimalType`, `NULL` and non-positive `price` / negative `freight` checks, allowing `freight_value == 0`) retained and expanded.
- **`[NEW ADDITION — PRODUCTION STANDARD]`**: Added enterprise lakehouse controls (schema contract, composite PK check, quarantine routing, FK checks, analytical flags, full lineage, reconciliation, and guarded write).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until a Dry Run completes cleanly and row/financial reconciliations pass.

## Section 2 — Configuration & Safety Controls `[NEW ADDITION — PRODUCTION STANDARD]`

Centralizes all paths, table targets, lineage metadata, and execution safety switches (`WRITE_ENABLED = False` by default) using relative OneLake shortcut paths instead of hardcoded workspace GUIDs.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Section 2: Configuration & Safety Controls
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from decimal import Decimal

WRITE_ENABLED = False        # SAFETY LOCK: flip to True only after Dry Run review
RUN_SELLER_FK_CHECK = True   # Cross-check seller_id against silver.sellers when present

BRONZE_PATH = "Files/bronze/sales/order_items/order_items_20261121_013211_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.order_items"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_order_items"
SILVER_SELLERS_TABLE = f"{SILVER_SCHEMA}.sellers"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.order_items"

EXPECTED_BRONZE_COLS = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
]

print(f"Configured BRONZE_PATH        : {BRONZE_PATH}")
print(f"Configured SILVER_TARGET      : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET  : {QUARANTINE_TARGET_TABLE}")
print(f"RUN_SELLER_FK_CHECK           : {RUN_SELLER_FK_CHECK}")
print(f"WRITE_ENABLED                 : {WRITE_ENABLED}")

## Section 3A — Teammate Original Ingestion `[ORIGINAL — COMMENTED FOR REVIEW]`

**What the teammate tried:** Reading the Bronze Parquet file using Pandas (`pd.read_parquet`) via an absolute `/lakehouse/default/...` path.

**Why it is commented out & replaced below:**
1. Loading Parquet into Pandas pulls the entire table into single-node **Driver memory**, bypassing Spark's distributed execution and Parquet schema metadata.
2. Converting Pandas back to Spark later (`df.astype(str)`) converts real `NULL`s into literal strings (`"nan"` / `"None"`), silently breaking `isNull()` quality checks.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 1 — COMMENTED FOR REVIEW]
# Preserved verbatim for transparency; replaced by native Spark Parquet read below.
# ==============================================================================
# import pandas as pd
# # Load data into pandas DataFrame from "/lakehouse/default/Files/bronze/sales/order_items/order_items_20261121_013211_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"
# df = pd.read_parquet("/lakehouse/default/Files/bronze/sales/order_items/order_items_20261121_013211_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet")
# display(df)
print("Skipped commented original Cell 1 (Pandas ingestion) — executing distributed Spark ingestion next.")

## Section 3B — Distributed Bronze Ingestion `[IMPROVED REPLACEMENT]`

Reads Bronze Parquet directly with PySpark from the relative `Files/bronze/...` shortcut and attaches `_source_file` via `F.input_file_name()`. Expected Bronze row count: **`112,650`**.

In [ ]:
# [IMPROVED REPLACEMENT] Direct distributed Spark Parquet ingestion
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze Order Items Row Count: {bronze_count:,}")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(5))

## Stage 1 — Schema Contract & Raw Null Baseline `[NEW ADDITION — PRODUCTION STANDARD]`

Verifies that all `7` expected Bronze columns exist, logs any unexpected extra columns (schema drift), and profiles raw `NULL`s before any casting occurs.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 1: Schema Contract & Baseline Profiling
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

raw_nulls = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
print("Stage 1 Schema Validation: PASSED")
display(raw_nulls)

## Stage 2A — Teammate Original Cleaning & Type Casting `[ORIGINAL — COMMENTED FOR REVIEW]`

- **What was RIGHT (`[KEPT / BUILT UPON]`):** Correctly chose fixed-point `DecimalType` instead of binary floating-point (`double`/`float`) for `price` and `freight_value`, and explicitly cast `shipping_limit_date` to `timestamp` and `order_item_id` to `int`.
- **Why parts needed updating:**
  1. `spark.createDataFrame(df.astype(str))` flattens types and hides `NULL`s behind `"nan"` strings.
  2. `F.regexp_replace(..., '"', "")` strips double quotes indiscriminately without any evidence of quote pollution in Parquet.
  3. Unprotected `.cast(...)` returns silent `NULL` on malformed values without counting or flagging cast failures.
  4. `DecimalType(10, 2)` is upgraded to `DecimalType(12, 2)` to align with downstream warehouse totals and derived `line_total_amount`.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 2 — COMMENTED FOR REVIEW]
# Good: Used DecimalType instead of double/float for financial columns.
# Replaced: Removed Pandas str coercion, unverified quote stripping, and silent casts.
# ==============================================================================
# from pyspark.sql import functions as F
# from pyspark.sql.types import DecimalType
# import pandas as pd
#
# if isinstance(df, pd.DataFrame):
#     df_spark = spark.createDataFrame(df.astype(str))
# else:
#     df_spark = df
#
# df_clean = (
#     df_spark
#     .withColumn("order_id", F.trim(F.regexp_replace(F.col("order_id"), '"', "")))
#     .withColumn("product_id", F.trim(F.regexp_replace(F.col("product_id"), '"', "")))
#     .withColumn("seller_id", F.trim(F.regexp_replace(F.col("seller_id"), '"', "")))
#     .withColumn("order_item_id", F.regexp_replace(F.col("order_item_id"), '"', "").cast("int"))
#     .withColumn(
#         "shipping_limit_date",
#         F.to_timestamp(F.regexp_replace(F.col("shipping_limit_date"), '"', ""))
#     )
#     .withColumn(
#         "price",
#         F.regexp_replace(F.col("price"), '"', "").cast(DecimalType(10, 2))
#     )
#     .withColumn(
#         "freight_value",
#         F.regexp_replace(F.col("freight_value"), '"', "").cast(DecimalType(10, 2))
#     )
# )
#
# df_clean.printSchema()
# display(df_clean.limit(10))
print("Skipped commented original Cell 2 — executing safe PySpark standardization next.")

## Stage 2B — Standardization, Safe `try_cast` & Line Total `[IMPROVED REPLACEMENT]`

1. Normalizes `order_id`, `product_id`, and `seller_id` via `F.lower(F.trim(...))`.
2. Applies `try_cast` for `order_item_id` (`INT`), `shipping_limit_date` (`TIMESTAMP`), `price` (`DECIMAL(12,2)`), and `freight_value` (`DECIMAL(12,2)`), tracking `_cast_failed` flags when a non-null raw value fails conversion.
3. Computes `line_total_amount = (price + freight_value)` in exact `DECIMAL(12,2)`.

In [ ]:
# [IMPROVED REPLACEMENT] Stage 2B: Standardization, Safe try_cast & Line Total
df_pre = (
    df_bronze
    .withColumn("_raw_order_item_id_str", F.col("order_item_id").cast("string"))
    .withColumn("_raw_ship_date_str", F.col("shipping_limit_date").cast("string"))
    .withColumn("_raw_price_str", F.col("price").cast("string"))
    .withColumn("_raw_freight_str", F.col("freight_value").cast("string"))
)

df_std = (
    df_pre
    .withColumn("order_id", F.lower(F.trim(F.col("order_id").cast("string"))))
    .withColumn("product_id", F.lower(F.trim(F.col("product_id").cast("string"))))
    .withColumn("seller_id", F.lower(F.trim(F.col("seller_id").cast("string"))))
    .withColumn("order_item_id", F.expr("try_cast(order_item_id AS INT)"))
    .withColumn("shipping_limit_date", F.expr("try_cast(shipping_limit_date AS TIMESTAMP)"))
    .withColumn("price", F.expr("try_cast(price AS DECIMAL(12,2))"))
    .withColumn("freight_value", F.expr("try_cast(freight_value AS DECIMAL(12,2))"))
    .withColumn("_cast_failed", (
        (F.col("_raw_order_item_id_str").isNotNull() & F.col("order_item_id").isNull()) |
        (F.col("_raw_ship_date_str").isNotNull() & F.col("shipping_limit_date").isNull()) |
        (F.col("_raw_price_str").isNotNull() & F.col("price").isNull()) |
        (F.col("_raw_freight_str").isNotNull() & F.col("freight_value").isNull())
    ))
    .withColumn("line_total_amount", (F.col("price") + F.col("freight_value")).cast(DecimalType(12, 2)))
)

cast_failures = df_std.filter(F.col("_cast_failed")).count()
print(f"Stage 2B Standardization complete. Cast failures detected: {cast_failures}")
df_std.select(*EXPECTED_BRONZE_COLS, "line_total_amount").printSchema()

## Stage 3 — Composite Primary Key `(order_id, order_item_id)` & Sequence Check `[NEW ADDITION — PRODUCTION STANDARD]`

- Enforces uniqueness of the true composite key `(order_id, order_item_id)` and checks sequence contiguity (`min=1`, `max=count` per order).
- Explicitly documents why `(order_id, product_id)` is **not** a unique key (`6,968` orders repeat the same `product_id` across sequential `order_item_id` lines to represent item quantity).

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 3: Composite PK & Sequence Continuity
exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()

dup_pk_df = (
    df_std
    .filter(F.col("order_id").isNotNull() & F.col("order_item_id").isNotNull())
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id", "order_item_id")
    .withColumn("_dup_pk", F.lit(True))
)

dup_pk_groups = dup_pk_df.count()

seq_gaps = (
    df_std
    .filter(F.col("order_id").isNotNull() & F.col("order_item_id").isNotNull())
    .groupBy("order_id")
    .agg(
        F.count("*").alias("cnt"),
        F.min("order_item_id").alias("min_id"),
        F.max("order_item_id").alias("max_id"),
    )
    .filter((F.col("min_id") != 1) | (F.col("max_id") != F.col("cnt")))
    .count()
)

df_std_pk = (
    df_std
    .join(dup_pk_df, on=["order_id", "order_item_id"], how="left")
    .withColumn("_dup_pk", F.coalesce(F.col("_dup_pk"), F.lit(False)))
)

print(f"Exact duplicate rows in Bronze          : {exact_dup_count}")
print(f"Duplicate (order_id, order_item_id) PKs : {dup_pk_groups} (Expected: 0)")
print(f"Orders with sequence gaps (not 1..N)    : {seq_gaps} (Expected: 0)")

## Stage 4 — Domain Validation, Seller FK Check & Quarantine Routing `[KEPT / BUILT UPON + NEW ADDITION]`

Expands the teammate's original price/freight check into full domain validation with explicit `dq_rejection_reason` tracking and non-destructive Quarantine routing:
1. Valid 32-char lowercase hex keys (`^[0-9a-f]{32}$`) for `order_id`, `product_id`, and `seller_id`.
2. `order_item_id >= 1`, `price > 0`, `freight_value >= 0` (preserving `freight_value = 0.00` free shipping), `shipping_limit_date IS NOT NULL`.
3. Zero cast failures (`_cast_failed = False`) and unique composite PK (`_dup_pk = False`).
4. Optional Referential Integrity check (`left_anti` join) against `silver.sellers` when `RUN_SELLER_FK_CHECK` is `True` and the table exists.

In [ ]:
# [KEPT / BUILT UPON + NEW ADDITION] Stage 4: Validation, Seller FK Check & Quarantine
hex32_regex = r"^[0-9a-f]{32}$"

if RUN_SELLER_FK_CHECK and spark.catalog.tableExists(SILVER_SELLERS_TABLE):
    df_sellers_ref = spark.table(SILVER_SELLERS_TABLE).select("seller_id").distinct()
    missing_seller_df = (
        df_std_pk.select("seller_id").distinct()
        .join(df_sellers_ref, on="seller_id", how="left_anti")
        .withColumn("_missing_seller_fk", F.lit(True))
    )
    missing_seller_cnt = missing_seller_df.count()
    df_checked = (
        df_std_pk
        .join(missing_seller_df, on="seller_id", how="left")
        .withColumn("_missing_seller_fk", F.coalesce(F.col("_missing_seller_fk"), F.lit(False)))
    )
    print(f"Referential Check vs {SILVER_SELLERS_TABLE}: {missing_seller_cnt} unmatched seller_id(s) (Expected: 0)")
else:
    df_checked = df_std_pk.withColumn("_missing_seller_fk", F.lit(False))
    print(f"Referential Check vs {SILVER_SELLERS_TABLE} skipped (table not found or check disabled).")

df_flagged = df_checked.withColumn(
    "dq_rejection_reason",
    F.concat_ws(
        " | ",
        F.when(~F.coalesce(F.col("order_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_ORDER_ID_HEX32")),
        F.when(~F.coalesce(F.col("product_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_PRODUCT_ID_HEX32")),
        F.when(~F.coalesce(F.col("seller_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_SELLER_ID_HEX32")),
        F.when(F.col("order_item_id").isNull() | (F.col("order_item_id") < 1), F.lit("INVALID_ORDER_ITEM_ID")),
        F.when(F.col("shipping_limit_date").isNull(), F.lit("INVALID_OR_NULL_SHIPPING_DATE")),
        F.when(F.col("price").isNull() | (F.col("price") <= F.lit(0)), F.lit("NON_POSITIVE_OR_NULL_PRICE")),
        F.when(F.col("freight_value").isNull() | (F.col("freight_value") < F.lit(0)), F.lit("NEGATIVE_OR_NULL_FREIGHT")),
        F.when(F.col("_cast_failed"), F.lit("TYPE_CAST_FAILED")),
        F.when(F.col("_dup_pk"), F.lit("DUPLICATE_COMPOSITE_PK")),
        F.when(F.col("_missing_seller_fk"), F.lit("MISSING_SELLER_FK")),
    )
)

df_valid = df_flagged.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_flagged.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(
        "order_id", "order_item_id", "product_id", "seller_id",
        "shipping_limit_date", "price", "freight_value",
        "dq_rejection_reason", "_source_file"
    )
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Valid Order Item Rows : {valid_count:,} (Expected: 112,650)")
print(f"Quarantined Rows      : {quarantine_count:,} (Expected: 0)")

## Stage 5 — Analytical Flags & Temporal Diagnostics `[NEW ADDITION — PRODUCTION STANDARD]`

Rather than silently altering or dropping valid orders with unusual business attributes, we expose explicit boolean flags for Gold analytics:
- `is_free_shipping`: `freight_value == 0.00` (Expected: **`383`** rows).
- `shipping_limit_outlier_flag`: `year(shipping_limit_date) > 2018` (Expected: **`4`** rows across `3` March–May 2017 orders whose `shipping_limit_date` was erroneously stamped in `2020` by `+1,051` to `+1,056` days; kept in Silver so order revenue reconciles exactly with `15,843,553.24`).

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 5: Business Flags & Anomaly Diagnostics
df_enriched = (
    df_valid
    .withColumn("is_free_shipping", F.col("freight_value") == F.lit(0))
    .withColumn("shipping_limit_outlier_flag", F.year(F.col("shipping_limit_date")) > 2018)
)

diag_row = df_enriched.agg(
    F.countDistinct("order_id").alias("distinct_orders"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.countDistinct("seller_id").alias("distinct_sellers"),
    F.sum(F.when(F.col("is_free_shipping"), 1).otherwise(0)).alias("free_shipping_rows"),
    F.sum(F.when(F.col("shipping_limit_outlier_flag"), 1).otherwise(0)).alias("year_gt_2018_outliers"),
    F.min("shipping_limit_date").alias("min_shipping_limit_date"),
    F.max("shipping_limit_date").alias("max_shipping_limit_date"),
).collect()[0]

print(f"Distinct Orders   : {diag_row['distinct_orders']:,} (Expected: 98,666)")
print(f"Distinct Products : {diag_row['distinct_products']:,} (Expected: 32,951)")
print(f"Distinct Sellers  : {diag_row['distinct_sellers']:,} (Expected: 3,095)")
print(f"Free Shipping Rows: {diag_row['free_shipping_rows']:,} (Expected: 383)")
print(f"2020 Outliers Flag: {diag_row['year_gt_2018_outliers']:,} (Expected: 4)")
print(f"Date Range        : {diag_row['min_shipping_limit_date']} -> {diag_row['max_shipping_limit_date']}")

display(
    df_enriched
    .filter(F.col("shipping_limit_outlier_flag"))
    .select("order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value")
)

## Stage 6 — Lineage Enrichment & Financial Reconciliation `[NEW ADDITION — PRODUCTION STANDARD]`

Projects the clean `14`-column Silver contract (`7` core columns + `3` analytical columns + `4` standardized lineage columns) and asserts:
1. **Row Conservation:** `bronze_count == silver_count + quarantine_count`
2. **Exact Financial Parity:** `SUM(price) == 13,591,643.70` and `SUM(freight_value) == 2,251,909.54` (Total `15,843,553.24`).

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 6: Full Lineage & Financial Reconciliation
df_silver = (
    df_enriched
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "line_total_amount",
        "is_free_shipping",
        "shipping_limit_outlier_flag",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Row reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)

fin_totals = df_silver.agg(
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight"),
    F.sum("line_total_amount").alias("sum_total"),
).collect()[0]

assert fin_totals["sum_price"] == Decimal("13591643.70"), f"Unexpected SUM(price): {fin_totals['sum_price']}"
assert fin_totals["sum_freight"] == Decimal("2251909.54"), f"Unexpected SUM(freight): {fin_totals['sum_freight']}"
assert fin_totals["sum_total"] == Decimal("15843553.24"), f"Unexpected SUM(line_total_amount): {fin_totals['sum_total']}"

df_silver.printSchema()
print(f"Stage 6 Row Reconciliation PASSED : bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")
print(f"Stage 6 Financial Parity PASSED   : price={fin_totals['sum_price']}, freight={fin_totals['sum_freight']}, total={fin_totals['sum_total']}")

## Section 5A — Teammate Original Validation & Unconditional Write `[ORIGINAL — COMMENTED FOR REVIEW]`

- **What was RIGHT (`[KEPT / BUILT UPON]`):** Checked per-column `NULL` counts, detected `price <= 0` and `freight_value < 0` while allowing `freight_value == 0`, and saved as Delta format.
- **Why parts needed updating:**
  1. Only printed `invalid_rows.count()` and then wrote the unfiltered `df_clean` anyway.
  2. Used `saveAsTable("order_items_cleaned")` directly in the default schema (`dbo`), diverging from the lakehouse `silver.<entity>` standard.
  3. Had no `WRITE_ENABLED` safety lock and no post-write Delta verification.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 3 — COMMENTED FOR REVIEW]
# Good: Checked NULLs, non-positive price, negative freight (kept freight==0), Delta write.
# Replaced: Enforced Quarantine split, silver.order_items schema, and WRITE_ENABLED gate.
# ==============================================================================
# from pyspark.sql import functions as F
#
# null_summary = df_clean.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(f"{c}_nulls")
#     for c in df_clean.columns
# ])
# display(null_summary)
#
# invalid_rows = df_clean.filter(
#     (F.col("price") < 0) | (F.col("freight_value") < 0) | (F.col("price") == 0)
# )
# print("عدد الصفوف ذات أسعار/شحن سالب أو سعر صفر:", invalid_rows.count())
#
# df_clean.write.mode("overwrite").format("delta").saveAsTable("order_items_cleaned")
# print("تم حفظ جدول order_items_cleaned بنجاح.")
print("Skipped commented original Cell 3 — executing guarded Silver Delta write next.")

## Section 5B — Guarded Delta Write (`silver.order_items`) `[IMPROVED REPLACEMENT]`

Creates `silver` schema if missing (`CREATE SCHEMA IF NOT EXISTS silver`) and writes `silver.order_items` (plus `silver._quarantine_order_items` only when `quarantine_count > 0`), gated strictly by `WRITE_ENABLED`.

In [ ]:
# [IMPROVED REPLACEMENT] Section 5B: Guarded Delta Write into silver.order_items
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification `[NEW ADDITION — PRODUCTION STANDARD]`

Reads back the persisted Delta table `silver.order_items` when `WRITE_ENABLED` is `True` and asserts row count, composite key uniqueness, analytical flags, and exact financial sums directly from storage.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup_pk = df_check.groupBy("order_id", "order_item_id").count().filter(F.col("count") > 1).count()
    chk_sums = df_check.agg(
        F.sum("price").alias("p"),
        F.sum("freight_value").alias("f"),
        F.sum("line_total_amount").alias("t"),
        F.sum(F.when(F.col("is_free_shipping"), 1).otherwise(0)).alias("free_cnt"),
        F.sum(F.when(F.col("shipping_limit_outlier_flag"), 1).otherwise(0)).alias("outlier_cnt"),
    ).collect()[0]

    assert chk_count == silver_count, f"Read-back row count mismatch: {chk_count} vs {silver_count}"
    assert chk_dup_pk == 0, f"Read-back duplicate composite PK found: {chk_dup_pk}"
    assert chk_sums["p"] == Decimal("13591643.70"), f"Read-back price sum mismatch: {chk_sums['p']}"
    assert chk_sums["f"] == Decimal("2251909.54"), f"Read-back freight sum mismatch: {chk_sums['f']}"

    print(
        f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup_pk}, "
        f"price={chk_sums['p']}, freight={chk_sums['f']}, total={chk_sums['t']}, "
        f"free_shipping={chk_sums['free_cnt']}, outliers_2020={chk_sums['outlier_cnt']}"
    )
    display(df_check.limit(5))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")